In [34]:
from langgraph.graph import StateGraph, START, END
import google.generativeai as genai
from typing import TypedDict
from dotenv import load_dotenv
import json
import os
import sys
import re
import pandas as pd

In [35]:
load_dotenv()

True

In [36]:
try:
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise ValueError("Error: GOOGLE_API_KEY environment variable not set.")
    genai.configure(api_key=api_key)
except Exception as e:
    print(f"Error: Unable to initialize Google GenAI client.")
    print(e)
    sys.exit(1)
safety_settings = {
            'HARM_CATEGORY_HARASSMENT': 'BLOCK_NONE',
            'HARM_CATEGORY_HATE_SPEECH': 'BLOCK_NONE',
            'HARM_CATEGORY_SEXUALLY_EXPLICIT': 'BLOCK_NONE',
            'HARM_CATEGORY_DANGEROUS_CONTENT': 'BLOCK_NONE',
        }
        
model = genai.GenerativeModel(
    'gemini-2.5-pro',
    safety_settings=safety_settings
)

In [37]:
class LLMState(TypedDict):
    riskReportFileName: str
    tabularReport: dict[str, any]
    postureScore: dict[str, float]
    

In [38]:
def calculate_posture_score(state: LLMState) -> dict:
    """_summary_

    Args:
        state (PostureState): _description_

    Returns:
        PostureState: _description_
    """
    report = state['tabularReport']
    print("tabularReport for scores: ", report)
    new_posture_scores = {}
    
    for key, item in report.items():
        if isinstance(item, dict):
            totalScore = 0
            sevScore = 0
            for sev, value in item.items():
                totalScore += value
                match sev:
                    case "s1":
                        sevScore = sevScore + value*10
                    case "s2":
                        sevScore = sevScore + value*8
                    case "s3":
                        sevScore = sevScore + value*6
                    case "s4":
                        sevScore = sevScore + value*4
                                            
            if totalScore > 0:
                new_posture_scores[key] = (sevScore/(totalScore*10))*100
            else:
                new_posture_scores[key] = 0.0
                                
    print("posture scores: ", new_posture_scores)
    
    # creating csv
    df_report = pd.DataFrame.from_dict(report, orient='index')
    df_score = pd.DataFrame.from_dict(new_posture_scores, orient='index', columns=['postureScore'])
    df_combined = df_report.join(df_score)
    df_combined.reset_index(inplace=True)
    df_combined.rename(columns={'index': 'hostName'}, inplace=True)
    combined_filename = "devices_posture_score.csv"
    df_combined.to_csv(combined_filename, index=False)
    
    print("Posture scores csv saved successfully!")
    
    return {"postureScore": new_posture_scores}

In [39]:
def load_json_data(filepath):
    """Safely loads a JSON file from the given path."""
    # This function is unchanged.
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return data
    except FileNotFoundError:
        print(f"Error: JSON file not found at path: {filepath}")
        return None
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON. Check file for formatting errors: {filepath}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while reading the file: {e}")
        return None

In [40]:
def generate_risk_report(state: LLMState)-> LLMState:
    output_filename = state['riskReportFileName']
    input_filename = "alerting.json"
    try:
        data_string = json.dumps(load_json_data(input_filename), indent=2)
        
        prompt = f"""
            You are a senior cyber security analyst. Your task is to generate a clear, structured risk report based on the JSON threat data provided.
            The report must be formatted using markdown (headings, bold, lists).
            ---JSON DATA---
            {data_string}
            ---END DATA---
            
            Please structure your report exactly as follows:
            # Cyber Threat Risk Report

            ## 1. Executive Summary
            (Provide a 2-3 sentence high-level overview of the most critical findings and affected hosts.)

            ## 2. Detailed Risk Analysis by Host
            (Create a separate sub-section for each unique `hostName` found in the data. For each host, provide the following three points based on the data.)

            ### Host: [hostname_1]
            * **Severity Breakdown:** (Summarize the count of threats by `severity`, e.g., s1: 2, s2: 5, s4: 1)
            * **Suspicious Processes:** (List the notable `processName` entries associated with these threats.)
            * **Key Detection Tags:** (List the `detectionTags` observed on this host.)

            ### Host: [hostname_2]
            * **Severity Breakdown:** ...
            * **Suspicious Processes:** ...
            * **Key Detection Tags:** ...
            (Add a new section for every other host)

            ## 3. Possible Remediation Steps
            (Based *only* on the analysis above, provide a bulleted list of actionable remediation steps. These steps should be generic but reference the types of findings.)

            * **Example:** "Investigate and terminate suspicious processes (e.g., `[processName]`) on affected hosts."
            * **Example:** "Quarantine hosts (e.g., `[hostname]`) exhibiting high-severity threats."
            * **Example:** "Update security policies to block or alert on `[detectionTag]`."

            Generate the report based *only* on the data provided.
            """
        
        response  = model.generate_content(prompt)
        
        report_content = response.text

        # --- Save the Report ---
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write(report_content)      
        print(f"\nSuccess! Report saved to: {output_filename}")
        
    except Exception as e:
        print(f"\nError during Google GenAI API call: {e}")
        
    return state

In [41]:
def generate_tabular_report(state: LLMState) -> dict:
    input_filename = "alerting.json"
    try:
        data_string = json.dumps(load_json_data(input_filename), indent=2)
        
        prompt = f"""
            You are a data analyst. Your task is to aggregate the provided JSON data.
            Based on the data, calculate the count of threat events, 
            categorized by severity level (s1, s2, s3, s4), for each unique 'hostName'.

            The source data is here:
            ---DATA_START---
            {data_string}
            ---DATA_END---

            Your response MUST be ONLY the resulting data formatted as a single, valid JSON object.
            Do not include markdown, backticks (```json), or any explanatory text.

            The JSON object should use the 'hostName' as the primary key.
            The value for each 'hostName' key should be another dictionary 
            containing the counts for "s1", "s2", "s3", and "s4".

            Example of the required output format:
            {{
            "host_A": {{ "s1": 5, "s2": 2, "s3": 0, "s4": 1 }},
            "host_B": {{ "s1": 10, "s2": 0, "s3": 1, "s4": 8 }}
            }}
        """
        
        response  = model.generate_content(prompt).text
        response = re.sub(r"```(json)?", "", response, flags=re.IGNORECASE)
        response = response.strip()
        data_dict = json.loads(response)
        print("state: ", data_dict)
        return {"tabularReport": data_dict}
        
    except Exception as e:
        print(f"\nError during Google GenAI API call: {e}")
    


In [42]:
# Graph Definition
graph  = StateGraph(LLMState)

# Node Definition
graph.add_node("risk_report", generate_risk_report)
graph.add_node("tabular_report", generate_tabular_report)
graph.add_node("calculate_posture_score", calculate_posture_score)

# Edge Definition
graph.add_edge(START, "risk_report")
graph.add_edge(START, "tabular_report")
graph.add_edge("tabular_report", "calculate_posture_score")
graph.add_edge("calculate_posture_score", END)
graph.add_edge("tabular_report", END)

# Compile Graph
workflow = graph.compile()

In [43]:
initial_state={"riskReportFileName": "risk_report.md"}
final_state = workflow.invoke(initial_state)

print(final_state)

E0000 00:00:1762992434.004767 2436770 http_proxy_mapper.cc:127] cannot parse value of 'http_proxy' env var. Error: INVALID_ARGUMENT: Could not parse 'scheme' from uri '10.44.64.190:9090'. Scheme must begin with an alpha character [A-Za-z].
E0000 00:00:1762992434.006183 2436771 http_proxy_mapper.cc:127] cannot parse value of 'http_proxy' env var. Error: INVALID_ARGUMENT: Could not parse 'scheme' from uri '10.44.64.190:9090'. Scheme must begin with an alpha character [A-Za-z].



Success! Report saved to: risk_report.md
state:  {'7242K25': {'s1': 22, 's2': 4, 's3': 0, 's4': 14}, '724W1124H2': {'s1': 12, 's2': 3, 's3': 0, 's4': 6}}
tabularReport for scores:  {'7242K25': {'s1': 22, 's2': 4, 's3': 0, 's4': 14}, '724W1124H2': {'s1': 12, 's2': 3, 's3': 0, 's4': 6}}
posture scores:  {'7242K25': 77.0, '724W1124H2': 80.0}
Posture scores csv saved successfully!
{'riskReportFileName': 'risk_report.md', 'tabularReport': {'7242K25': {'s1': 22, 's2': 4, 's3': 0, 's4': 14}, '724W1124H2': {'s1': 12, 's2': 3, 's3': 0, 's4': 6}}, 'postureScore': {'7242K25': 77.0, '724W1124H2': 80.0}}
